# UC-UW-4 — Async DWH Join Export via the existing `dwh_join` Process

**Goal:** Drive the existing `dwh_join` Process (id `dwh_join`, scope `COLLECTION`, async-only) to run a complex BigQuery query and export a joined GeoJSONL file to the catalog bucket.

The query is the canonical fixture from the test suite — `tests/dynastore/extensions/dwh/test_dwh_repro.py:59-68`:

```
dwh_query="SELECT * FROM dwh"
dwh_join_column="join_col"
join_column="join_col"
```

The fake `SELECT * FROM dwh` matches the existing dwh-extension test path — no real BQ table needed for the local round-trip. Substitute a real query against a sandbox project for an end-to-end remote demo.

**Comparison aside (last cell):** the same payload also drives the synchronous `/dwh/join` endpoint when the UI wants the join inline rather than as an exported job.


In [ ]:
import json
import os
import time
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)
RUN_ID = os.environ.get("RUN_ID", uuid.uuid4().hex[:8])
CATALOG_ID = os.environ.get("CATALOG_ID", f"uw_{RUN_ID}")
COLLECTION_ID = os.environ.get("COLLECTION_ID", f"col_{RUN_ID}")

# REMOTE_ONLY heuristic: pub/sub-driven flows do not fire against localhost.
IS_LOCAL = "localhost" in BASE_URL or "127.0.0.1" in BASE_URL
PUBSUB_ENABLED = not IS_LOCAL

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"

client = httpx.Client(base_url=BASE_URL, headers=headers, timeout=120.0)

print(f"BASE_URL      : {BASE_URL}")
print(f"CATALOG_ID    : {CATALOG_ID}")
print(f"COLLECTION_ID : {COLLECTION_ID}")
print(f"RUN_ID        : {RUN_ID}")
print(f"AUTH          : {'token set' if TOKEN else 'anonymous'}")
print(f"PUBSUB        : {'enabled (remote)' if PUBSUB_ENABLED else 'disabled (local)'}")


## Step 0 — Discover the catalog bucket

In [ ]:
# Step 0 — Discover the catalog's bucket; build destination_uri under it.
r = client.get(f"/gcp/buckets/catalogs/{CATALOG_ID}")
assert r.status_code == 200, f"bucket lookup: {r.text}"
BUCKET = r.json().get("bucket_id") or r.json().get("bucket_name")
print(f"BUCKET = gs://{BUCKET}")


## Step 1 — Build the request (canonical test fixture)

In [ ]:
# Step 1 — Build the request. Verbatim from the test fixture for the query bits.
DWH_PROJECT_ID = os.environ.get("DWH_PROJECT_ID", "p")
DESTINATION_URI = f"gs://{BUCKET}/{COLLECTION_ID}/exports/{{job_id}}.geojsonl"

inputs = {
    "dwh_project_id": DWH_PROJECT_ID,
    "dwh_query": "SELECT * FROM dwh",
    "catalog": CATALOG_ID,
    "collection": COLLECTION_ID,
    "dwh_join_column": "join_col",
    "join_column": "join_col",
    "output_format": "geojson",
    "with_geometry": True,
    "destination_uri": DESTINATION_URI,
}
print(json.dumps(inputs, indent=2))


## Step 2 — Submit async (Prefer: respond-async)

In [ ]:
# Step 2 — Submit the async dwh_join Process. Prefer: respond-async ⇒ 202 + job_id.
r = client.post(
    f"/processes/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/processes/dwh_join/execution",
    content=json.dumps({"inputs": inputs, "outputs": {}}),
    headers={**headers, "Prefer": "respond-async"},
)
print(f"submit: {r.status_code} — {r.text[:300]}")
assert r.status_code in (200, 201, 202), f"dwh_join exec: {r.text}"
receipt = r.json()
job_id = receipt.get("jobID") or receipt.get("job_id")
assert job_id, f"no job id in receipt: {receipt}"
print(f"job_id = {job_id}")


## Step 3 — Poll for terminal status

In [ ]:
# Step 3 — Poll until terminal status.
status = "accepted"
for attempt in range(30):
    r = client.get(f"/processes/jobs/{job_id}")
    assert r.status_code == 200, f"poll: {r.text}"
    body = r.json()
    status = body.get("status", "")
    print(f"attempt {attempt+1}: status={status}")
    if status in ("successful", "failed", "dismissed"):
        break
    time.sleep(min(2 ** attempt, 15))
print(f"terminal status: {status}")


## Step 4 — Fetch result

In [ ]:
# Step 4 — Fetch results link or failure detail.
r = client.get(f"/processes/jobs/{job_id}/results")
print(f"results: {r.status_code}")
print(json.dumps(r.json(), indent=2)[:1000] if r.status_code == 200 else r.text[:300])


## Step 5 — Comparison: synchronous `/dwh/join`

In [ ]:
# Step 5 — Comparison: same payload, synchronous /dwh/join endpoint.
sync_payload = {
    "dwh_project_id": DWH_PROJECT_ID,
    "dwh_query": "SELECT * FROM dwh",
    "catalog": CATALOG_ID,
    "collection": COLLECTION_ID,
    "dwh_join_column": "join_col",
    "join_column": "join_col",
    "output_format": "geojson",
    "with_geometry": True,
}
r = client.post("/dwh/join", content=json.dumps(sync_payload))
print(f"sync /dwh/join: {r.status_code}")
print(r.text[:600])
client.close()
